# Episode 26: Constrained decoding: structured LLM output

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep026-constrained-decoding.ipynb)

Nothing is trained. Before each token inside a query's name, our agent's graph role masks the tokens that would take the name off a list, and we measure two lists, the graph's names and the task's words.

[The video](https://youtu.be/KypK6nrP-nE) · [The episode's page](https://mvgpt.nenuadrian.com/ep026-constrained-decoding/) · [The docs](https://nenuadrian.github.io/mvGPT/constrained-decoding/) · [The code at `ep026/end`](https://github.com/nenuadrian/mvGPT/tree/ep026/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep026/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("graph-agent")

## smollerGraph
The graph the agent asks, as in episode 25: smollerGraph at its episode 10, loaded with the
mammals and birds on Wikidata, then `sg serve` on port 8474, where mvGPT looks for it.

In [ ]:
if not os.path.isdir("/content/smollerGraph"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/smollerGraph /content/smollerGraph
!pip install --quiet cmake ninja
!cmake -S /content/smollerGraph -B /content/smollerGraph/build -G Ninja > /dev/null && cmake --build /content/smollerGraph/build --target sg -- --quiet
!rm -f /content/smollerGraph/wikidata.sg && /content/smollerGraph/build/sg load /content/smollerGraph/wikidata.sg /content/smollerGraph/data/wikidata.txt
sg = start("/content/smollerGraph/build/sg serve /content/smollerGraph/wikidata.sg 8474", "http://127.0.0.1:8474/api/names?prefix=R")

## The tests
The tokens a name allows, anything outside a name, double quotes for an apostrophe, a letter
split across tokens, names from the task, and the mask.

In [ ]:
!python -m pytest -q tests/test_constrain.py

## The Okapi, a token at a time
Episode 25's agent, from the Hub, writing the graph role's query for the Okapi: first kept to
the names the graph has, then to the words of the task.

In [ ]:
!python constrain.py checkpoints/graph-agent/model.pt "How much does the Okapi weigh?" /content/smollerGraph/data/wikidata.txt
!python constrain.py checkpoints/graph-agent/model.pt "How much does the Okapi weigh?"

## The data
Episode 21's tasks (they need FineWeb-Edu's last file for the web pages), episode 24's
questions and episode 25's tasks, whose held-out sets the evaluation asks.

In [ ]:
!python -c "from data import WEB_SHARDS, download_shard; download_shard(WEB_SHARDS - 1)"  # FineWeb-Edu's last file, which pretraining never read
!python agent_data.py
!python graph_data.py /content/smollerGraph/data/wikidata.txt
!python graph_agent_data.py /content/smollerGraph/data/wikidata.txt

## Measuring it
The graph role on episode 24's held-out questions, free, with the graph's names and with the
task's words, then the agent on episode 25's tasks, free and with the task's words.

In [ ]:
!python constrain_eval.py checkpoints/graph-agent/model.pt /content/smollerGraph/data/wikidata.txt

## Asking it
The Okapi with the task's words, and with the graph's names, which walk it to the Oak
Titmouse; the Komodo dragon, which the graph hasn't got; a name with an apostrophe; and the
Okapi against the Baikal Seal.

In [ ]:
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Okapi than the Lion?" --names task
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Okapi than the Lion?" --names graph --facts /content/smollerGraph/data/wikidata.txt
!python agent.py checkpoints/graph-agent/model.pt "Is the Komodo dragon a mammal or a bird?" --names graph --facts /content/smollerGraph/data/wikidata.txt
!python agent.py checkpoints/graph-agent/model.pt "Is the Komodo dragon a mammal or a bird?" --names task
!python agent.py checkpoints/graph-agent/model.pt "Which family does Turner's Eremomela belong to?" --names task
!python agent.py checkpoints/graph-agent/model.pt "How much heavier is the Okapi than the Baikal Seal?" --names task

Next: [episode 27, Speculative decoding from scratch](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep027-speculative-decoding.ipynb)